In [ ]:
import pandas as pd
import numpy as np
import random
import math

def calculate_suitability(row, w0=0.7, w1=0.2, w2=-0.1, w3=-0.1):
    return w0 * row['ALLSKY_SFC_SW_DWN'] + w1 * row['ALLSKY_KT'] + w2 * row['CLOUD_AMT'] + w3 * row['PRECTOTCORR']

def optimize_solar_sites(df, budget=20, max_iterations=1000, initial_temp=1000, cooling_rate=0.003, verbose=False):
    df = df.copy()
    df['suitability_score'] = df.apply(calculate_suitability, axis=1)

    num_points = budget
    current_solution = np.random.choice(df.index, num_points, replace=False)
    current_cost = -df.loc[current_solution, 'suitability_score'].sum()
    best_solution = current_solution.copy()
    best_cost = current_cost
    temperature = initial_temp

    for i in range(max_iterations):
        new_solution = current_solution.copy()
        d1_idx = random.randint(0, num_points - 1)
        d2 = random.choice([idx for idx in df.index if idx not in new_solution])
        new_solution[d1_idx] = d2

        new_cost = -df.loc[new_solution, 'suitability_score'].sum()

        if new_cost < current_cost or random.uniform(0, 1) < math.exp((current_cost - new_cost) / temperature):
            current_solution = new_solution.copy()
            current_cost = new_cost

            if current_cost < best_cost:
                best_solution = current_solution.copy()
                best_cost = current_cost

        temperature *= (1 - cooling_rate)

        if verbose and i % 100 == 0:
            print(f"Iteration {i + 1}, Current Fitness: {-current_cost}, Best Fitness: {-best_cost}")

    best_points = df.loc[best_solution]
    best_points.to_csv('best_20_points_optimized.csv', index=False)
    print("Best 20 grid points saved to 'best_20_points_optimized.csv'")

    # Print detailed grid point info
    print("\nSelected Grid Points:")
    print(best_points[['lat', 'lon', 'suitability_score']])

    best_fitness = -best_cost / num_points
    return best_points, best_fitness

def run_optimize_solar_sites():
    df_input = pd.read_csv('nasa_annual_summary_l2_normalized_scaled.csv')
    best_points, best_fitness = optimize_solar_sites(
        df_input,
        budget=20,
        max_iterations=1000,
        initial_temp=1000,
        cooling_rate=0.003,
        verbose=True
    )
    print("Best fitness (average score):", best_fitness)
    return best_points, best_fitness

if __name__ == "__main__":
    run_optimize_solar_sites()


In [ ]:
import pandas as pd
import numpy as np
import random
import math

def calculate_suitability(row, w0=0.7, w1=0.2, w2=-0.1, w3=-0.1):
    return w0 * row['ALLSKY_SFC_SW_DWN'] + w1 * row['ALLSKY_KT'] + w2 * row['CLOUD_AMT'] + w3 * row['PRECTOTCORR']

def run_pso_solar_sites():
    df_input = pd.read_csv('nasa_annual_summary_l2_normalized_scaled.csv')
    df_input['suitability_score'] = df_input.apply(calculate_suitability, axis=1)

    N = len(df_input)
    swarmsize = 100
    maxiter = 100
    c1 = 2.0
    c2 = 2.0
    w = 0.7
    budget = 20

    # Initialize particles: each particle is an array of selected indices
    particles = [np.random.choice(N, budget, replace=False) for _ in range(swarmsize)]
    velocities = [np.zeros(budget) for _ in range(swarmsize)]

    def fitness_function(indices):
        return df_input.loc[indices, 'suitability_score'].sum()

    pbest = particles.copy()
    pbest_fitness = np.array([fitness_function(p) for p in particles])
    gbest = particles[np.argmax(pbest_fitness)].copy()
    gbest_fitness = np.max(pbest_fitness)

    for iteration in range(maxiter):
        for i in range(swarmsize):
            fitness = fitness_function(particles[i])
            if fitness > pbest_fitness[i]:
                pbest[i] = particles[i].copy()
                pbest_fitness[i] = fitness
                if fitness > gbest_fitness:
                    gbest = particles[i].copy()
                    gbest_fitness = fitness

        for i in range(swarmsize):
            # Update velocity (conceptually, swap indices toward best)
            new_indices = particles[i].copy()
            swap_idx = random.randint(0, budget - 1)
            if gbest[swap_idx] not in new_indices:
                new_indices[swap_idx] = gbest[swap_idx]
            particles[i] = np.unique(new_indices)
            if len(particles[i]) < budget:
                additional = np.random.choice([idx for idx in range(N) if idx not in particles[i]], budget - len(particles[i]), replace=False)
                particles[i] = np.concatenate([particles[i], additional])

        if iteration % 10 == 0:
            print(f"Iteration {iteration + 1}, Best Fitness: {gbest_fitness}")

    best_points = df_input.loc[gbest]
    best_points.to_csv('best_pso_points.csv', index=False)
    print("Best PSO grid points saved to 'best_pso_points.csv'")
    print("\nSelected Grid Points:")
    print(best_points[['lat', 'lon', 'suitability_score']])
    best_fitness = gbest_fitness / budget
    print("Best fitness (average score):", best_fitness)
    return best_points, best_fitness

if __name__ == "__main__":
    run_pso_solar_sites()
